<!-- scprint2-nb-header:begin -->
# Get started: prepare scPRINT-2

One-off setup: download the ontologies into lamindb and load a scPRINT-2 checkpoint. Run this once per environment before any of the other example notebooks.

## Contents

  - [1. Initialize the lamindb instance](#1-initialize-the-lamindb-instance)
  - [2. Imports](#2-imports)
  - [3. Populate the ontology (one-off, ~5 min)](#3-populate-the-ontology-one-off-5-min)
  - [4. (Optional) Repair an older checkpoint](#4-optional-repair-an-older-checkpoint)
- [Set the path of the checkpoint you want to repair before running the cells below.](#set-the-path-of-the-checkpoint-you-want-to-repair-before-running-the-cells-below)
  - [5. Load a scPRINT-2 model (you'll do this in every notebook)](#5-load-a-scprint-2-model-youll-do-this-in-every-notebook)
    - [Reconciling the model's gene vocabulary with your ontology](#reconciling-the-models-gene-vocabulary-with-your-ontology)

> _Run `notebooks/prepare_scprint2.ipynb` once before any other example notebook._

<!-- scprint2-nb-header:end -->

## 1. Initialize the lamindb instance

`scdataloader` and `scPRINT-2` use [lamindb](https://lamin.ai) to manage all single-cell ontologies (gene IDs, cell types, diseases, tissues, …) in a local SQLite store. **Run this once per environment**; the cell is commented because you typically execute it in your shell, but uncommenting it works inside Jupyter too.

- `--storage ./testdb` is where the lamindb database lives on disk.
- `--modules bionty` enables the ontology module (without it, the gene/cell-type lookups below will fail).


In [ ]:
# ! lamin init --storage ./testdb --name test --modules bionty
# ! lamin connect anonymous/testdb

## 2. Imports

The two functions we need from `scdataloader`:

- `populate_my_ontology` downloads the ontology records into the local lamindb instance.
- `_adding_scbasecamp_genes` adds the extra gene records that the pretraining dataset (scBaseCamp) uses on top of cellxgene's default vocabulary.
- `load_genes` is used later when sanity-checking the model against the current ontology.


In [1]:
from scdataloader.utils import _adding_scbasecamp_genes, populate_my_ontology, load_genes
import os.path
import urllib.request
import torch

%load_ext autoreload
%autoreload 2

## 3. Populate the ontology (one-off, ~5 min)

Downloads the ontology terms scPRINT-2 cares about. By default we keep only the two species shipped in the public checkpoints (human + mouse) and the two biological sexes; other axes (cell type, disease, assay, …) are populated with their full vertebrate clade. Uncomment the keyword arguments to restrict further (faster but breaks anything outside the restricted scope).

After this step the local lamindb is self-contained: no internet access is needed for the remaining notebooks.


In [ ]:
populate_my_ontology(
    organisms_clade=["vertebrates"],
    sex=["PATO:0000384", "PATO:0000383"],
    organisms=["NCBITaxon:10090", "NCBITaxon:9606"],
    # celltypes=None,
    # ethnicities=None,
    # assays=None,
    # tissues=None,
    # diseases=None,
    # dev_stages=None,
)
_adding_scbasecamp_genes()

## 4. (Optional) Repair an older checkpoint

**Skip this section unless** loading a checkpoint raises a `KeyError` on a `expr_encoder.encoder.X` weight or a `None` label-decoder entry.

Some pre-2026 scPRINT/scPRINT-2 checkpoints contain extra weight names that no longer exist in the current `scprint2` package (renamed during the encoder refactor), and `None` values inside `label_decoders` that newer pytorch-lightning refuses to unpickle. The three cells below rewrite those keys in place. Set `model_checkpoint_file` to the path of your checkpoint before running.


```python
# Set the path of the checkpoint you want to repair before running the cells below.
model_checkpoint_file = "../../models/old-checkpoint.ckpt"
```


In [ ]:
m = torch.load(
    model_checkpoint_file, map_location=torch.device("cpu"), weights_only=False
)

Rewrite the legacy keys to match the current `scprint2` module names:


In [ ]:
m["hyper_parameters"]["label_decoders"] = {
    k: {u: j if j is not None else "None" for u, j in v.items()}
    for k, v in m["hyper_parameters"]["label_decoders"].items()
}
rn = {
    "expr_encoder.encoder.2.weight": "expr_encoder.encoder.1.weight",
    "expr_encoder.encoder.2.bias": "expr_encoder.encoder.1.bias",
    "expr_encoder.encoder.6.weight": "expr_encoder.encoder.5.weight",
    "expr_encoder.encoder.6.bias": "expr_encoder.encoder.5.bias",
    "expr_decoder.fc.2.weight": "expr_decoder.fc.1.weight",
    "expr_decoder.fc.2.bias": "expr_decoder.fc.1.bias",
    "expr_decoder.fc.6.weight": "expr_decoder.fc.5.weight",
    "expr_decoder.fc.6.bias": "expr_decoder.fc.5.bias",
}
m["state_dict"] = {k if k not in rn else rn[k]: v for k, v in m["state_dict"].items()}

m["hyper_parameters"].pop("checkpointing")
m["hyper_parameters"].pop("residual_in_fp32")
m["hyper_parameters"].pop("fused_dropout_add_ln")
m["hyper_parameters"].pop("checkpointing")
m["hyper_parameters"].pop("fused_mlp")
m["hyper_parameters"].pop("fused_bias_fc")
m["hyper_parameters"].pop("drop_path_rate")
m["hyper_parameters"].pop("class_compression")
m["hyper_parameters"].pop("depth_atinput")
m["hyper_parameters"].pop("cell_transformer_layers")
m["hyper_parameters"].pop("residual_in_fp32")

Save the repaired checkpoint back to disk (overwrites the original — make a copy first if you want to keep the legacy version):


In [ ]:
torch.save(m, model_checkpoint_file)

## 5. Load a scPRINT-2 model (you'll do this in every notebook)

The block below is the canonical way to load a public scPRINT-2 checkpoint:

1. Download the checkpoint from HuggingFace if it isn't cached locally.
2. Load it via `scPRINT2.load_from_checkpoint` (a pytorch-lightning wrapper).
3. Move it to CUDA if available, otherwise stay on CPU in float32 (half-precision on CPU is brittle).

Available public checkpoints on [jkobject/scPRINT on HuggingFace](https://huggingface.co/jkobject/scPRINT):
- `small-v2.ckpt` — fastest, for the example notebooks (used here).
- `medium-v2.ckpt`, `large-v2.ckpt`, `vlarge-v2.ckpt` — paper-grade checkpoints.


In [ ]:
LOC = "../../models/"  # "../../../"
ckpt_path = os.path.join(LOC, "small-v2.ckpt")
if not os.path.exists(ckpt_path):
    url = "https://huggingface.co/jkobject/scPRINT/resolve/main/small-v2.ckpt"
    urllib.request.urlretrieve(url, ckpt_path)

In [ ]:
model = scPRINT2.load_from_checkpoint(
    ckpt_path,
    precpt_gene_emb=None,
    gene_pos_file=None,
)
if not torch.cuda.is_available():
    model = model.to(torch.float32)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

### Reconciling the model's gene vocabulary with your ontology

Public ontologies (cellxgene, Ensembl) occasionally retire gene IDs between scPRINT-2 releases. The check below detects genes the checkpoint was trained on but that no longer exist in the local ontology, and removes them from the model in-place. **You should run this once after loading any checkpoint** to avoid silent downstream failures when a removed gene appears in your input data.


In [ ]:
# in some cases the gene ontology has changed too much since I trained the model,
# so I need to remove the genes that are not in the ontology anymore from the model
missing = set(model.genes) - set(load_genes(model.organisms).index)
if len(missing) > 0:
    print(
        "Warning: some genes missmatch exist between model and ontology: solving...",
    )
    model._rm_genes(missing)